# DiploMind: 3×3 dataset generation (language × depth), v2

Every article is processed. There are no topic filters and no quality rejects.

Each raw Urdu article becomes one English summary, then 9 cells:
- **depths:** easy, medium, hard (English written first, from the summary)
- **languages:** `en`, `urdish`, `ur` (Urdish and Urdu are conversions of the English cell at the same depth)

Urdish = Urdu written in English (Roman) letters with English words mixed in, the way people text in Pakistan.

**What changed from v1:**
- No IR / sensitive filter and no quality checks: every article with text is processed and kept.
- The gold examples are sent as separate chat turns, so the model no longer copies them into its answer.
- Every depth must keep every fact (v2 first try shrank 'hard' to one sentence).
- A separate, stronger model (`LANG_MODEL`) writes the Urdu and Urdish, and the Urdu translation sees the original article for correct names.
- Numbers are kept as written (v1 turned "September" into 9).
- Output files are named `*_v2_*`, so v1 results don't get in the way.

**3 machines:** set a different `MACHINE_ID` on each (section 3). Re-running the run cell resumes where it stopped.

## 1. Install

In [ ]:
!pip install -q openai

## 2. API key
Paste your W&B key below. Clear this cell before sharing the notebook.
On Kaggle you can leave it empty and add a secret named `WANDB_API_KEY` instead (Add-ons → Secrets).

In [ ]:
WANDB_API_KEY = ""
WANDB_PROJECT = "m-ibrahim094131-fast-nuces/test"

import os
if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
print("API key set" if WANDB_API_KEY else "No API key yet: paste it above")

API key set


## 3. Settings

In [27]:
import os, re, csv, json, time, glob, random, hashlib, threading, collections
from concurrent.futures import ThreadPoolExecutor

MACHINE_ID   = "m1"            # m1 / m2 / m3: different on every machine
ARTICLES_DIR = "/kaggle/input/datasets/mibrahim094131/articles"      # folder with the scraped articles, e.g. /kaggle/input/<dataset>/articles_m1
OUT_ROOT     = "dataset_out"   # on Kaggle: /kaggle/working/dataset_out

MODEL      = "meta-llama/Llama-3.3-70B-Instruct"   # summary + English easy/medium/hard
LANG_MODEL = "deepseek-ai/DeepSeek-V3.1"           # English -> Urdish / Urdu (Llama 3.3 is weak at Urdu)
# Other W&B options worth trying for LANG_MODEL (see the comparison cell in section 10):
#   "google/gemma-4-31B-it", "moonshotai/Kimi-K2.6", "Qwen/Qwen3.8-27B"
BASE_URL = "https://api.inference.wandb.ai/v1"

LANGS  = ["en", "urdish", "ur"]
DEPTHS = ["easy", "medium", "hard"]

MAX_ARTICLES = None   # None = every article in the folder; e.g. 10 for a test run
WORKERS      = 4      # articles processed in parallel; lower it if you hit rate limits
EXAMPLE_RATE = 0.3    # share of articles whose EASY cells get one everyday example
TEMPERATURE  = 0.3

# Optional automatic split across machines. Leave NUM_SHARDS = 1 if each machine has its own folder.
NUM_SHARDS  = 1
SHARD_INDEX = 0

OUT_DIR = os.path.join(OUT_ROOT, MACHINE_ID)
os.makedirs(OUT_DIR, exist_ok=True)
RAW_PATH  = os.path.join(OUT_DIR, f"grid_raw_v2_{MACHINE_ID}.jsonl")    # every processed article (resume file)
PREP_PATH = os.path.join(OUT_DIR, f"prep_cache_v2_{MACHINE_ID}.jsonl")  # cached summaries
print("Machine:", MACHINE_ID, "| writing to", OUT_DIR)

Machine: m1 | writing to dataset_out/m1


## 4. Load the model (W&B Inference client) and test it

In [28]:
from openai import OpenAI

assert WANDB_API_KEY, "Paste the API key in section 2 first."
client = OpenAI(base_url=BASE_URL, api_key=WANDB_API_KEY, project=WANDB_PROJECT)
def is_reasoning(m):   # reasoning models need a bigger token budget
    return any(k in m.lower() for k in ("gpt-oss", "kimi", "glm", "minimax"))

for m in (MODEL, LANG_MODEL):
    r = client.chat.completions.create(model=m, max_tokens=200 if is_reasoning(m) else 5,
                                       messages=[{"role": "user", "content": "Reply with OK."}])
    print(f"{m}: {r.choices[0].message.content!r}")

meta-llama/Llama-3.3-70B-Instruct: 'OK'
deepseek-ai/DeepSeek-V3.1: 'OK.'


## 5. Gold samples (few-shot examples)
Replace these with your own checked samples. They are sent to the model as earlier chat turns (example question → example answer), so they guide the style without ending up in the output.
- `GOLD_SOURCES`: short English texts, like the summaries the pipeline makes.
- `GOLD_DEPTH`: the same texts rewritten at each depth, **in the same order** as `GOLD_SOURCES`.
- `GOLD_EASY_META`: for each easy rewrite, the terms it explains and whether it has an everyday example (`yes`/`no`).
- `GOLD_LANG`: (English, converted) pairs for Urdish and Urdu.

In [29]:
GOLD_SOURCES = [
    "The UN Security Council has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say its impact depends on how strongly major powers support it.",
    "A new round of trade talks between the US and China begins next week. Both countries will try to narrow differences over tariffs and export restrictions. Observers say a major breakthrough is unlikely, though continued talks are a positive sign.",
    "The European Union has announced new economic sanctions on a country. Assets of some of its banks will be frozen and energy exports will be limited. The affected country called the decision unfair and threatened countermeasures.",
]

GOLD_DEPTH = {
    "easy": [
        "The UN Security Council is the UN group that decides on peace and security. It passed a resolution, which is a formal decision, asking both sides to stop fighting right away. This is called a ceasefire. Thirteen members voted yes and two chose not to vote. Experts say the decision will only work if the big powers support it.",
        "The US and China will start a new round of trade talks next week. They disagree about tariffs, which are extra taxes on goods brought in from other countries. They also disagree about export restrictions, which are limits on what a country can sell abroad. For example, if a tariff makes imported chocolate cost more, shoppers may buy local chocolate instead. Experts do not expect a big result. Still, they say it is a good sign that the talks continue.",
        "The EU announced new sanctions on a country. Sanctions are penalties on trade and money, used to pressure a government. Some of the country's banks will have their assets frozen, which means the banks cannot use that money. Energy exports, such as oil and gas, will also be limited. The country called this unfair and threatened to respond.",
    ],
    "medium": [
        "The UN Security Council, the UN body responsible for international peace and security, has passed a resolution calling for an immediate ceasefire. Thirteen members voted in favour, while two did not vote. Diplomatic experts say the resolution's impact will depend on how strongly the major powers support it.",
        "The United States and China will begin a new round of trade talks next week. They will try to narrow their differences over tariffs, which are taxes on imported goods, and over export restrictions. Observers expect no major breakthrough, but they see the continuation of talks as a positive sign.",
        "The European Union has announced new economic sanctions, meaning penalties on trade and finance, against a country. Some of the country's banks will have their assets frozen, and its energy exports will be limited. The affected country called the decision unfair and threatened to take countermeasures.",
    ],
    "hard": [
        "The UN Security Council adopted a resolution demanding an immediate ceasefire, carried by thirteen votes with two members not participating in the vote. Diplomatic analysts assess that its implementation will hinge on the degree of political backing extended by the major powers.",
        "A new round of US-China trade negotiations opens next week, focused on narrowing divergences over tariff regimes and export restrictions between the two economies. Observers anticipate limited substantive progress, although they regard the continuation of the dialogue itself as a constructive signal.",
        "The EU has imposed a new package of economic sanctions on the target state, including asset freezes on several of its banks and limits on its energy exports. The targeted government has rejected the measures as unjust and signalled possible retaliatory countermeasures.",
    ],
}

GOLD_EASY_META = [
    ("Security Council, resolution, ceasefire", "no"),
    ("tariffs, export restrictions", "yes"),
    ("sanctions, freeze assets", "no"),
]

GOLD_LANG = {
    "urdish": [
        ("The UN Security Council has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say its impact depends on how strongly major powers support it.",
         "UN Security Council ne ek resolution pass kiya hai jis mein fareeqain se foran ceasefire ka mutalba kiya gaya hai. Resolution ke haq mein terah votes aaye jabke do members ne voting mein hissa nahi liya. Diplomatic experts ke mutabiq is resolution par amal hona is baat par depend karta hai ke bari powers ise kitni support karti hain."),
        ("The EU announced new sanctions on a country. Sanctions are penalties on trade and money, used to pressure a government. Some of the country's banks will have their assets frozen, which means the banks cannot use that money. Energy exports, such as oil and gas, will also be limited. The country called this unfair and threatened to respond.",
         "European Union ne ek mulk par nayi sanctions lagayi hain. Sanctions un pabandiyon ko kehte hain jo paise aur business par lagayi jati hain taake kisi hukumat par dabao dala jaye. Is mulk ke kuch banks ke assets freeze kiye jayein ge, yaani woh banks woh paisa istemal nahi kar sakein ge. Tail aur gas jaisi energy ki exports par bhi hadd lagegi. Us mulk ne kaha ke yeh na-insafi hai aur jawab dene ki dhamki di hai."),
    ],
    "ur": [
        ("The UN Security Council has passed a resolution calling on the parties for an immediate ceasefire. Thirteen members voted in favour and two did not vote. Diplomatic experts say its impact depends on how strongly major powers support it.",
         "اقوام متحدہ کی سلامتی کونسل نے ایک قرارداد منظور کی ہے جس میں فریقین سے فوری جنگ بندی کا مطالبہ کیا گیا ہے۔ قرارداد کے حق میں تیرہ ووٹ آئے جبکہ دو ارکان نے ووٹنگ میں حصہ نہیں لیا۔ سفارتی ماہرین کے مطابق اس قرارداد پر عمل درآمد کا انحصار اس بات پر ہے کہ بڑی طاقتیں اس کی کتنی حمایت کرتی ہیں۔"),
        ("The UN Security Council is the UN group that decides on peace and security. It passed a resolution, which is a formal decision, asking both sides to stop fighting right away. This is called a ceasefire. Thirteen members voted yes and two chose not to vote. Experts say the decision will only work if the big powers support it.",
         "اقوام متحدہ کی سلامتی کونسل اقوام متحدہ کا وہ ادارہ ہے جو امن اور سلامتی کے فیصلے کرتا ہے۔ اس نے ایک قرارداد منظور کی، یعنی ایک باقاعدہ فیصلہ، جس میں دونوں فریقوں سے کہا گیا کہ فوراً لڑائی بند کریں۔ اسے جنگ بندی کہتے ہیں۔ تیرہ ارکان نے حق میں ووٹ دیا اور دو نے ووٹ نہیں دیا۔ ماہرین کا کہنا ہے کہ یہ فیصلہ تب ہی کارگر ہوگا جب بڑی طاقتیں اس کی حمایت کریں گی۔"),
    ],
}

for d in DEPTHS:
    assert len(GOLD_DEPTH[d]) == len(GOLD_SOURCES), f"GOLD_DEPTH['{d}'] needs one rewrite per GOLD_SOURCES entry"
assert len(GOLD_EASY_META) == len(GOLD_SOURCES), "GOLD_EASY_META needs one entry per GOLD_SOURCES entry"
print("Gold samples OK:", len(GOLD_SOURCES), "depth examples,",
      {l: len(v) for l, v in GOLD_LANG.items()}, "language pairs")

Gold samples OK: 3 depth examples, {'urdish': 2, 'ur': 2} language pairs


## 6. Prompts
Each request is a chat: the rules go in the system message, each gold example is a separate user → assistant turn, and the real article is the last user message.

In [30]:
SYS_PREP = """You summarise Urdu news articles in English.
Reply in exactly this format and nothing else:
SUMMARY: 3 to 5 plain English sentences with the main facts of the article. Keep every name, number and date exactly as in the article. Use standard English spellings of names and titles. Add nothing that is not in the article and do not comment on the article.
TERMS: 1 to 3 political, legal, economic or international-relations concepts from your summary that a beginner would not know (for example: governor's rule, sanctions, constituency, IMF bailout). Never pick ordinary words (protest, gunfire, police, school, injured). Comma-separated, lowercase. Write none if there are none."""

SYS_DEPTH = {
    "easy": """You rewrite news for a complete beginner who knows nothing about politics or world affairs.
Rules:
- One paragraph of plain prose with short sentences (about 8 to 15 words each). Split long sentences into several short ones.
- Cover EVERY sentence of the Text: keep every name, place, number, date and fact, exactly as written. Do not leave anything out and do not add facts.
- It will be LONGER than the Text, because you also explain the listed Terms.
- Explain each listed Term once, in a short plain phrase, where it first appears (for example: "governor's rule, which means the federal government runs the province directly"). Do not explain any other word.
- If Example is yes, add ONE short everyday example of what one of the Terms means in real life. If Example is no, add no example.
- Rewrite ONLY the Text in the latest message. Output only the rewrite.""",
    "medium": """You rewrite news for an interested general reader, like a university student from another field.
Rules:
- One paragraph of clear prose (sentences of about 12 to 22 words).
- Keep technical terms; add a short clarifying phrase only for terms a general reader may not know.
- No examples or analogies.
- Cover EVERY sentence of the Text: keep every name, place, number, date and fact, exactly as written. Do not leave anything out and do not add facts.
- About the same length as the Text.
- Rewrite ONLY the Text in the latest message. Output only the rewrite.""",
    "hard": """You rewrite news in the register of an analyst writing for specialists.
Rules:
- One paragraph of formal, precise prose (sentences of about 15 to 30 words), using specialist vocabulary and do not explain terms.
- No examples, analogies or opinions of your own.
- Cover EVERY sentence of the Text: keep every name, place, number, date and fact, exactly as written. Do not leave anything out and do not add facts.
- About the same length as the Text: change the vocabulary and sentence style, NOT the amount of information. Never compress it into one or two sentences.
- Rewrite ONLY the Text in the latest message. Output only the rewrite.""",
}

SYS_LANG = {
    "urdish": """You convert English text to Urdish: Urdu written in English (Roman) letters, mixed with English words, the way Pakistanis write in everyday text messages (hai, hain, ka, ki, mein, aur, ke liye...).
Rules:
- Same content, same order, same difficulty, same explanations. Do not simplify, add or drop anything.
- Keep English for technical and political terms (sanctions, ceasefire, Security Council).
- Use Pakistani Urdu words (hukumat, mulk, faisla, aman, afsar, pur-aman, tashaddud), never Hindi words (sarkar, shanti, shant, vishay, adhikari, hinsak, samasya).
- Spell names exactly as in the English. No Urdu script and no Devanagari. Keep numbers exactly as in the English.
- Convert ONLY the text in the latest message. Output only the converted text.""",
    "ur": """You translate English text into natural, standard Urdu in Urdu script, as written in Pakistani newspapers.
Rules:
- Same content, same order, same difficulty, same explanations. Do not simplify, add or drop anything.
- Use established Urdu terms (پابندیاں، جنگ بندی، سلامتی کونسل). If no Urdu term is common, write the English term in Urdu script.
- Keep numbers exactly as in the English. For names of people and places, use the spelling in the original Urdu article if one is given.
- Write only in Urdu script. Never use Devanagari (Hindi) letters or Hindi words.
- Translate ONLY the English text in the latest message. Output only the Urdu text.""",
}

def depth_user(depth, text, terms="none", ex="no"):
    if depth == "easy":
        return f"Text: {text}\nTerms: {terms}\nExample: {ex}"
    return f"Text: {text}"

def depth_messages(depth, text, terms, ex):
    msgs = [{"role": "system", "content": SYS_DEPTH[depth]}]
    for i, s in enumerate(GOLD_SOURCES):
        t, e = GOLD_EASY_META[i] if depth == "easy" else ("none", "no")
        msgs.append({"role": "user", "content": depth_user(depth, s, t, e)})
        msgs.append({"role": "assistant", "content": GOLD_DEPTH[depth][i]})
    msgs.append({"role": "user", "content": depth_user(depth, text, terms, ex)})
    return msgs

def lang_messages(lang, english, urdu_original=""):
    msgs = [{"role": "system", "content": SYS_LANG[lang]}]
    for en, tr in GOLD_LANG[lang]:
        msgs.append({"role": "user", "content": en})
        msgs.append({"role": "assistant", "content": tr})
    if lang == "ur" and urdu_original:
        # the original article helps with correct Urdu spellings of names, places and terms
        english = (f"Original Urdu article (only for spellings of names, places and terms):\n{urdu_original}\n\n"
                   f"English text to translate:\n{english}")
    msgs.append({"role": "user", "content": english})
    return msgs

# Sentences from the gold answers. If one shows up in an output, the model copied an example.
GOLD_SENTS = {s.strip().lower() for txt in [*GOLD_SOURCES, *sum(GOLD_DEPTH.values(), []),
                                            *[t for p in GOLD_LANG.values() for pair in p for t in pair]]
              for s in re.split(r"[.!?۔]", txt) if len(s.split()) >= 6}
print("Prompts ready")

Prompts ready


## 7. Load the articles folder
Reads every `.jsonl`, `.json` and `.txt` file under `ARTICLES_DIR`, including the output of `scrape_urdu_articles.py`.

In [31]:
def make_article_id(url, text):
    key = url.strip() if url else text.strip()[:2000]
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

def _as_article(d, fallback_url):
    text = d.get("text") or d.get("content") or d.get("body") or ""
    url = d.get("url") or d.get("link") or fallback_url
    return {"url": url, "text": text, "title": d.get("title", "")}

def load_articles(folder):
    arts = []
    files = sorted(glob.glob(os.path.join(folder, "**", "*"), recursive=True))
    for path in files:
        ext = os.path.splitext(path)[1].lower()
        rel = os.path.relpath(path, folder)
        try:
            if ext == ".jsonl":
                for n, line in enumerate(open(path, encoding="utf-8")):
                    if line.strip():
                        arts.append(_as_article(json.loads(line), f"{rel}#{n}"))
            elif ext == ".json":
                data = json.load(open(path, encoding="utf-8"))
                items = data if isinstance(data, list) else [data]
                for n, d in enumerate(items):
                    arts.append(_as_article(d, f"{rel}#{n}"))
            elif ext == ".txt":
                arts.append({"url": rel, "text": open(path, encoding="utf-8").read(), "title": ""})
        except Exception as e:
            print(f"  skipped {rel}: {type(e).__name__}: {e}")
    out, seen = [], set()
    for a in arts:
        if not a["text"].strip():
            continue
        a["article_id"] = make_article_id(a["url"], a["text"])
        if a["article_id"] in seen:          # same article twice in THIS folder: process once
            continue
        seen.add(a["article_id"])
        if NUM_SHARDS > 1 and int(a["article_id"][:8], 16) % NUM_SHARDS != SHARD_INDEX:
            continue
        out.append(a)
    return out[:MAX_ARTICLES] if MAX_ARTICLES else out

assert os.path.isdir(ARTICLES_DIR), f"Folder not found: {ARTICLES_DIR}"
ARTICLES = load_articles(ARTICLES_DIR)
print(f"{len(ARTICLES)} articles loaded for machine {MACHINE_ID}")
if ARTICLES:
    a = ARTICLES[0]
    print("First one:", a["url"], "|", a["text"][:200].replace("\n", " "), "...")

272 articles loaded for machine m1
First one: https://www.bbc.com/urdu/articles/cmp3gn5nrpddo | پاکستان کے صوبہ خیبرپختونخوا میں ایمرجنسی اور گورنر راج کے خلاف ضلع خیبر کے علاقے باڑہ میں گذشتہ روز (13 اکتوبر) ہونے والے پُرتشدد مظاہروں کے بعد آج (بدھ) بھی بازار بند رہے جبکہ اس علاقے میں پولیس اور ...


## 8. LLM helpers

In [32]:
LOCK = threading.Lock()

def clean(t):
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip()
    t = re.sub(r"^\s*(Urdish|Urdu|English|Rewrite|Output|Text)\s*:\s*", "", t, flags=re.I)
    return re.sub(r"\s*\n+\s*", " ", t.strip().strip('"')).strip()

def llm(messages, max_tokens=900, model=None):
    model = model or MODEL
    mt = max_tokens * 4 if is_reasoning(model) else max_tokens
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=model, max_tokens=mt, temperature=TEMPERATURE,
                                               messages=messages)
            return r.choices[0].message.content or ""
        except Exception as e:
            print(f"   [llm retry {attempt+1}] {type(e).__name__}: {str(e)[:80]}")
            time.sleep(2 ** attempt)
    raise RuntimeError("llm failed after retries")

def copied_gold(t):
    tl = t.lower()
    return [s for s in GOLD_SENTS if s in tl]

def remove_gold(t):
    """Last resort: drop any copied example sentence from the output."""
    parts = re.split(r"(?<=[.!?۔])\s+", t)
    return " ".join(p for p in parts if not copied_gold(p)).strip()

DEVANAGARI = re.compile(r"[\u0900-\u097F]")
HINDI_WORDS = re.compile(r"\b(sarkar|shanti|shant|vishay|adhikari|hinsak|samasya|upyog|rashtra|suraksha|yuddh|mantri|pradhan)\b", re.I)

def problem(t, kind, source=""):
    """What is wrong with an output, or None. Used only to ask the model ONE more time; nothing is ever rejected."""
    if not t:
        return "Your answer was empty."
    if copied_gold(t):
        return "You copied the earlier examples. Give the answer for my latest text only."
    if DEVANAGARI.search(t):
        return "You used Hindi (Devanagari) letters. Write it again without any Devanagari."
    if kind == "urdish" and HINDI_WORDS.search(t):
        return f"You used the Hindi word '{HINDI_WORDS.search(t).group(0)}'. Use Pakistani Urdu words instead."
    if kind in ("medium", "hard") and source and len(t.split()) < 0.6 * len(source.split()):
        return "Your answer left out facts. Cover every sentence of the Text, with every name, place and number."
    if kind == "easy" and source and len(t.split()) < 0.8 * len(source.split()):
        return "Your answer left out facts. Cover every sentence of the Text, with every name, place and number."
    return None

def generate(messages, kind, source="", model=None, max_tokens=900):
    """One call; if something is clearly wrong, ask once more with the reason. The output is always kept."""
    t = clean(llm(messages, max_tokens, model))
    why = problem(t, kind, source)
    if why:
        t2 = clean(llm(messages + [{"role": "assistant", "content": t or "(empty)"},
                                    {"role": "user", "content": why}], max_tokens, model))
        if t2 and (problem(t2, kind, source) is None or not t):
            t = t2
    return remove_gold(t) if copied_gold(t) else t

def parse_prep(t):
    t = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).replace("**", "")
    m = re.search(r"SUMMARY\s*:\s*(.*?)(?:\n\s*TERMS\s*:|$)", t, re.S | re.I)
    summary = clean(m.group(1) if m else t)
    m = re.search(r"TERMS\s*:\s*(.*)", t, re.I)
    terms = [x.strip(" .'\"").lower() for x in (m.group(1) if m else "").split(",")]
    terms = [x for x in terms if x and x not in ("none", "n/a") and x.isascii()][:4]
    return summary, terms

def chunk(text, max_words=350):
    """Article body without junk lines, cut to max_words."""
    paras = [p.strip() for p in text.split("\n") if p.strip() and not re.search(r"BBC|بی بی سی|©|\.com", p)]
    words = " ".join(paras).split()
    return " ".join(words[:max_words]) if len(words) >= 30 else None

print("Helpers ready")

Helpers ready


## 9. Pipeline: one article → 9 cells
1. Summarise the Urdu article in English and pick the terms to explain.
2. Write the English cell at each depth from the summary.
3. Convert each English cell to Urdish and Urdu.

No filters: an article is only skipped if it has no text, or if the API keeps failing (those are retried on the next run).

In [33]:
PREP_CACHE = {}
if os.path.exists(PREP_PATH):
    for line in open(PREP_PATH, encoding="utf-8"):
        r = json.loads(line)
        PREP_CACHE[r.pop("article_id")] = r

def prep(art):
    aid = art["article_id"]
    if aid in PREP_CACHE:
        return PREP_CACHE[aid]
    body = chunk(art["text"])
    if not body:
        return {"reject": "no_text"}
    summary, terms = parse_prep(llm([{"role": "system", "content": SYS_PREP},
                                     {"role": "user", "content": body}], 600))
    res = {"summary": summary, "terms": terms}
    with LOCK:
        PREP_CACHE[aid] = res
        with open(PREP_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({"article_id": aid, **res}, ensure_ascii=False) + "\n")
    return res

def make(art):
    base = {"article_id": art["article_id"], "url": art["url"], "machine": MACHINE_ID}
    p = prep(art)
    if p.get("reject"):
        return {**base, "reject": p["reject"]}
    use_ex = int(art["article_id"][:8], 16) % 100 < EXAMPLE_RATE * 100
    row = {**base, "input": p["summary"], "terms": p["terms"], "use_example": use_ex, "cells": {}, "reject": None}
    for depth in DEPTHS:
        row["cells"][f"en_{depth}"] = generate(
            depth_messages(depth, p["summary"], ", ".join(p["terms"]) or "none", "yes" if use_ex else "no"),
            kind=depth, source=p["summary"])
    urdu_original = " ".join((chunk(art["text"]) or "").split()[:250])
    for depth in DEPTHS:
        for lang in ("urdish", "ur"):
            row["cells"][f"{lang}_{depth}"] = generate(
                lang_messages(lang, row["cells"][f"en_{depth}"], urdu_original),
                kind=lang, model=LANG_MODEL, max_tokens=1400 if lang == "ur" else 1100)
    return row

print("Pipeline ready")

Pipeline ready


## 10. Pilot: run a couple of articles and read the output
Nothing is saved here. Check the summary and all 9 cells before the full run.

In [34]:
PILOT_N = 2
for art in ARTICLES[:PILOT_N]:
    row = make(art)
    print("=" * 100)
    print("URL:", row["url"], "| skipped:", row.get("reject"))
    if row.get("input"):
        print("\nSUMMARY:", row["input"], "\nTERMS:", row["terms"], "| example:", row["use_example"])
        for d in DEPTHS:
            for l in LANGS:
                print(f"\n[{l} / {d}]\n{row['cells'][f'{l}_{d}']}")

URL: https://www.bbc.com/urdu/articles/cmp3gn5nrpddo | skipped: None

SUMMARY: Protests against the emergency and governor's rule in Pakistan's Khyber Pakhtunkhwa province continued for a second day in the Bara area of Khyber district, with markets remaining closed and a heavy police and FC presence in the area. According to Peshawar police, 21 people, including police personnel, were injured in clashes between police and PTI workers on 13 October, while 62 people were taken into custody. The protests took place in the constituency of recently removed Chief Minister Suhail Afridi. At least five injured people with gunshot wounds were brought to the Type D Dogra hospital in Bara. CCPO Peshawar Dr Mian Saeed visited Bara to review security arrangements and appealed to people to remain peaceful. 
TERMS: ['emergency', "governor's rule", 'constituency', 'gunfire'] | example: False

[en / easy]
There are protests in Pakistan's Khyber Pakhtunkhwa province. The protests are against the emergen

### 10b. Which model writes the best Urdu / Urdish?
Converts one English cell with a few W&B models side by side. Read them (or ask a native speaker), then set `LANG_MODEL` in section 3 to the winner and re-run sections 3, 4 and 9 before the full run.

In [36]:
CANDIDATES = ["deepseek-ai/DeepSeek-V3.1", "google/gemma-4-31B-it", "moonshotai/Kimi-K2.6",
              "Qwen/Qwen3.8-27B", "meta-llama/Llama-3.3-70B-Instruct","deepseek-ai/DeepSeek-V4.1-Flash"]
art = ARTICLES[0]
p = prep(art)
english = generate(depth_messages("easy", p["summary"], ", ".join(p["terms"]) or "none", "no"),
                   kind="easy", source=p["summary"])
original = " ".join((chunk(art["text"]) or "").split()[:250])
print("ENGLISH (medium):\n", english)
for m in CANDIDATES:
    for lang in ("urdish", "ur"):
        try:
            out = generate(lang_messages(lang, english, original), kind=lang, model=m, max_tokens=1400)
        except Exception as e:
            out = f"(failed: {type(e).__name__})"
        print(f"\n----- {m} / {lang}\n{out}")

ENGLISH (medium):
 There are protests in Pakistan's Khyber Pakhtunkhwa province. The protests are against the emergency, which is a special situation where the government has more control. They are also against the governor's rule, which means the federal government runs the province directly. The protests are in the Bara area of Khyber district. Markets are closed. There are many police and FC, which is a security force, in the area. On 13 October, police and PTI workers clashed. PTI workers are supporters of a political party. 21 people were hurt, including police. 62 people were taken into custody. The protests are in the area where Suhail Afridi was in charge. He was the Chief Minister, but he was removed. At least five people with gunshot wounds were taken to the hospital. Gunshot wounds are injuries from gunfire, which is when a gun is fired. CCPO Peshawar Dr Mian Saeed visited Bara. He checked the security arrangements. He asked people to remain peaceful.

----- deepseek-ai/Deep

## 11. Full run (resumable)
Every processed article is appended to `grid_raw_v2_<machine>.jsonl`. If the session dies, run this cell again and it continues.

In [ ]:
done = {}
if os.path.exists(RAW_PATH):
    for line in open(RAW_PATH, encoding="utf-8"):
        r = json.loads(line)
        if not str(r.get("reject") or "").startswith("error"):   # API errors are retried
            done[r["article_id"]] = r

todo = [a for a in ARTICLES if a["article_id"] not in done]
print(f"{len(ARTICLES)} articles | {len(done)} already done | {len(todo)} to process | model = {MODEL}")

def safe(art):
    try:
        return make(art)
    except Exception as e:
        return {"article_id": art["article_id"], "url": art["url"], "machine": MACHINE_ID,
                "reject": f"error:{type(e).__name__}"}

B = WORKERS * 4
t0 = time.time()
with ThreadPoolExecutor(WORKERS) as ex, open(RAW_PATH, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), B):
        for r in ex.map(safe, todo[i:i + B]):
            done[r["article_id"]] = r
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()
        counts = collections.Counter(r.get("reject") or "done" for r in done.values())
        print(f"  {min(i + B, len(todo))}/{len(todo)} | {time.time() - t0:.0f}s | " +
              ", ".join(f"{k}={v}" for k, v in counts.most_common()))

## 12. Export this machine's dataset
Writes in `OUT_DIR`:
- `grid_<machine>.csv`: every article as 9 rows (one per language × depth). The adapter notebook merges these.
- `review_<machine>.csv`: about 10% of articles (all 9 cells) for a native speaker to read, with `ok` / `notes` columns.

In [ ]:
def write_csv(path, header, rows):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:   # utf-8-sig so Urdu opens correctly in Excel
        w = csv.writer(f, quoting=csv.QUOTE_ALL)
        w.writerow(header)
        w.writerows(rows)

rows = {}
for line in open(RAW_PATH, encoding="utf-8"):
    r = json.loads(line)
    rows[r["article_id"]] = r
good = [r for r in rows.values()
        if not r.get("reject") and all(r["cells"].get(f"{l}_{d}") for l in LANGS for d in DEPTHS)]
skipped = collections.Counter(r.get("reject") for r in rows.values() if r.get("reject"))

grid_path = os.path.join(OUT_DIR, f"grid_{MACHINE_ID}.csv")
write_csv(grid_path,
          ["article_id", "url", "machine", "language", "depth", "input", "output", "terms", "use_example"],
          [[r["article_id"], r["url"], r["machine"], l, d, r["input"], r["cells"][f"{l}_{d}"],
            "; ".join(r["terms"]), r["use_example"]]
           for r in good for d in DEPTHS for l in LANGS])

sample = random.Random(7).sample(good, min(len(good), max(5, round(len(good) * 0.1))))
write_csv(os.path.join(OUT_DIR, f"review_{MACHINE_ID}.csv"),
          ["article_id", "language", "depth", "input", "output", "ok (y/n)", "notes"],
          [[r["article_id"], l, d, r["input"], r["cells"][f"{l}_{d}"], "", ""]
           for r in sample for d in DEPTHS for l in LANGS])

print(f"Articles in dataset: {len(good)} -> {len(good) * 9} rows in {grid_path}")
print("Skipped:", dict(skipped) or "none")